# Model Comparison

This notebook tunes and compares Logistic Regression, XGBoost, and Histogram-based Gradient Boosting. Shared loading, splitting, and dataset-building helpers are imported from `model_functions.py` so this notebook stays focused on model comparison.

---
## 1) Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from scripts.model_functions import (
    build_dataset_with_groups,
    learn_training_ai_excess_words,
    load_corpus,
)

# Conservative OpenMP settings for macOS/XGBoost stability.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


---
## 2) Load Aligned Data
As explained in `main.ipynb`, the human and ai data are aligned to prevent data leakage. We split the data into a training and test set, after which we run K-fold crossvalidation on the trainingset. Since we're doing K-fold crossvalidation, we need to find a way to keep the human and ai articles aligned. For that we create groups where each group hold one human-article and its ai-counterpart, using `build_dataset_with_groups`. We then use `GroupKfold` to run K-fold crossvalidation, while keeping both members of a group in the same split.

We then convert all our trainingdata, since models like XGBoost work best with numpy arrays.

In [2]:
human_train, human_test = load_corpus("human_articles")
ai_train, ai_test = load_corpus("ai_articles")

In [3]:
ai_excess_words, ai_excess_scores = learn_training_ai_excess_words(
    ai_train,
    human_train,
    top_n=2600,
    min_count=3,
)

X_train_raw, Y_train, train_groups = build_dataset_with_groups(
    human_train,
    ai_train,
    ai_excess_words,
)

X_test_raw, Y_test, test_groups = build_dataset_with_groups(
    human_test,
    ai_test,
    ai_excess_words,
)

X_train_raw = np.asarray(X_train_raw, dtype=np.float32)
X_test_raw = np.asarray(X_test_raw, dtype=np.float32)
Y_train = np.asarray(Y_train, dtype=bool)
Y_test = np.asarray(Y_test, dtype=bool)
train_groups = np.asarray(train_groups)

print(f"Training article pairs: {len(human_train)}")
print(f"Testing article pairs:  {len(human_test)}")
print(f"Training windows: {len(X_train_raw)}")
print(f"Testing windows:  {len(X_test_raw)}")


KeyboardInterrupt: 

---
## 3) Models And Parameter Grids
Since we're runnning a standard scaler before our logistic regression model, we need to create a pipeline that scales all iterations of Kfold. Keeping the following functions simple, we do the same to our other models.

Our grids are based on personal testing, where we started with a large range and slowly narrowed it down based on the best results. We evaluate our grid search on Macro F1.

We do have slight data leakage during validation, with the "learned ai marker" words being based on the whole training dataset. Idealy it would be based on the training data, minus the validation data. However, since this is an equal advantage to all validation splits, and we're not using it for final evaluation, it shouldn't interfere with the results. It will be important to note that the crossvalidation macro F1 might be a little inflated.

In [ ]:
cv = GroupKFold(n_splits=5)

# Use macro F1 for tuning so neither class silently dominates the comparison.
scoring = "f1_macro"

LR_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
LR_param_grid = {
    "logisticregression__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "logisticregression__class_weight": [None, "balanced"],
}

XGB_model = make_pipeline(
    XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="exact",
        device="cpu",
        n_jobs=1,
        random_state=42,
    )
)
XGB_param_grid = {
    "xgbclassifier__n_estimators": [100, 200],
    "xgbclassifier__max_depth": [2, 3, 4],
    "xgbclassifier__learning_rate": [0.03, 0.05, 0.1],
    "xgbclassifier__subsample": [0.8, 1.0],
    "xgbclassifier__colsample_bytree": [0.8, 1.0],
    "xgbclassifier__reg_lambda": [1, 5, 10],
}

HistB_model = make_pipeline(
    HistGradientBoostingClassifier(random_state=42)
)
HistB_param_grid = {
    "histgradientboostingclassifier__max_iter": [100, 200, 300],
    "histgradientboostingclassifier__learning_rate": [0.03, 0.05, 0.1],
    "histgradientboostingclassifier__max_leaf_nodes": [7, 15, 31],
    "histgradientboostingclassifier__l2_regularization": [0.0, 0.1, 1.0],
    "histgradientboostingclassifier__min_samples_leaf": [10, 20, 30],
}


---
## 4) Tuning Helpers

In [ ]:
def run_grid_search(name, model, param_grid):
    """
    Runs grid search of [model] on [param_grid]. Then stores the results, to later be printed.
    """
    grid = GridSearchCV(
        model,
        param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )
    grid.fit(X_train_raw, Y_train, groups=train_groups)

    Y_pred = grid.best_estimator_.predict(X_test_raw)
    result = {
        "name": name,
        "grid": grid,
        "best_params": grid.best_params_,
        "cv_f1_macro": grid.best_score_,
        "test_accuracy": grid.best_estimator_.score(X_test_raw, Y_test),
        "test_f1_macro": f1_score(Y_test, Y_pred, average="macro"),
        "test_f1_ai": f1_score(Y_test, Y_pred, pos_label=False),
        "test_f1_human": f1_score(Y_test, Y_pred, pos_label=True),
        "classification_report": classification_report(
            Y_test,
            Y_pred,
            target_names=["AI", "Human"],
        ),
    }
    return result


def print_result(result):
    """
    Prints forms of evaluation of the [result].
    """
    print(result["name"])
    print("Best params:", result["best_params"])
    print(f"CV macro F1:    {result['cv_f1_macro']:.4f}")
    print(f"Test accuracy:  {result['test_accuracy']:.4f}")
    print(f"Test macro F1:  {result['test_f1_macro']:.4f}")
    print(f"Test AI F1:     {result['test_f1_ai']:.4f}")
    print(f"Test Human F1:  {result['test_f1_human']:.4f}")
    print()
    print(result["classification_report"])


---
## 5) Logistic Regression

In [ ]:
LR_result = run_grid_search("Logistic Regression", LR_model, LR_param_grid)
print_result(LR_result)

---
## 6) XGBoost

In [ ]:
XGB_result = run_grid_search("XGBoost", XGB_model, XGB_param_grid)
print_result(XGB_result)

---
## 7) Histogram-Based Gradient Boosting

In [ ]:
HistB_result = run_grid_search("Histogram-Based Gradient Boosting", HistB_model, HistB_param_grid)
print_result(HistB_result)

---
## 8) Compare Models

In [ ]:
results = [LR_result, XGB_result, HistB_result]

for result in sorted(results, key=lambda item: item["test_f1_macro"], reverse=True):
    print(
        f"{result['name']:<36} "
        f"CV macro F1={result['cv_f1_macro']:.4f}  "
        f"Test macro F1={result['test_f1_macro']:.4f}  "
        f"AI F1={result['test_f1_ai']:.4f}  "
        f"Human F1={result['test_f1_human']:.4f}"
    )

After running and tuning all the models, the models all seem to perform very similarly. With the models test macro F1 ranging from 0.89 (LR) to 0.90 (XGBoost and HBGB), and the untuned LR model having a macro F1 of 0.89 as well (found in `main.ipynb`), it seems that from these models there isn't too big of a difference. Since Logistic Regression has the benefit of having one coefficient per feature and an intercept, it is more interpretable than the other 2 models. We therefore decided to stick to Logistic Regression.